# Tarea 3 — PyTorch

**Nombre del integrante:** Gabriel Jimenez

**Cédula del integrante:** 32001385

> Enunciado, rúbrica y nombres de los archivos `.pth` a entregar: ver `README.md` de esta carpeta.
> Antes del commit: *Restart & Run All* debe terminar sin errores y quedar en el menú.

## 1. Imports y constantes

In [ ]:
import os

import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from torchvision import datasets

NOTEBOOK_ID = 2  # N of pytorch_fashion_N.ipynb (used in the .pth file names)
DATA_DIR = "data"  # gitignored
INPUT_SIZE = 784
NUM_CLASSES_ORIGINAL = 10
CLASS_NAMES_ORIGINAL = ["T-shirt/top", "Trouser", "Pullover", "Dress", "Coat",
                        "Sandal", "Shirt", "Sneaker", "Bag", "Ankle boot"]
CLASS_NAMES_SIMPLIFIED = ["Top", "Bottom", "Footwear", "Bag"]
# Fashion MNIST label -> simplified label (Top, Bottom, Footwear, Bag)
SIMPLIFIED_MAP = torch.tensor([0, 1, 0, 1, 0, 2, 0, 2, 3, 2])
BATCH_SIZE = 128
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## 2. Datos (Fashion MNIST original y clases simplificadas)

Los datos se descargan en `./data` (ignorada por git). Las imágenes quedan en el rango `[0, 1]`,
igual que `transforms.ToTensor()`, sin normalización fuera del modelo.

In [ ]:
def load_split(train):
    dataset = datasets.FashionMNIST(root=DATA_DIR, train=train, download=True)
    images = dataset.data.float().div(255).unsqueeze(1)  # same values as ToTensor()
    return images, dataset.targets


train_images, train_labels = load_split(train=True)
test_images, test_labels = load_split(train=False)


def get_labels(labels, classification):
    if classification == "simplified":
        return SIMPLIFIED_MAP[labels]
    return labels


def get_class_names(classification):
    return CLASS_NAMES_SIMPLIFIED if classification == "simplified" else CLASS_NAMES_ORIGINAL

## 3. Modelos (rectangular y convolucional)

- **Rectangular:** todas las capas ocultas tienen el mismo número de neuronas.
- **Convolucional:** el número de neuronas de la capa oculta `i` (de `L`) se calcula por
  interpolación lineal entre la entrada (784) y la salida:
  `neurons(i) = round(784 + (salida - 784) * i / (L + 1))`.
  Con `L = 2` y 10 salidas: 526 → 268.

In [ ]:
def layer_sizes(network_type, hidden_layers, neurons_per_layer, num_outputs):
    """Widths of every layer, from the input (784) to the output."""
    if network_type == "convolutional":
        hidden = [round(INPUT_SIZE + (num_outputs - INPUT_SIZE) * i / (hidden_layers + 1))
                  for i in range(1, hidden_layers + 1)]
    else:
        hidden = [neurons_per_layer] * hidden_layers
    return [INPUT_SIZE] + hidden + [num_outputs]


def build_model(sizes, batch_norm=False, dropout=0.0):
    layers = [nn.Flatten()]
    for i, (n_in, n_out) in enumerate(zip(sizes[:-1], sizes[1:])):
        layers.append(nn.Linear(n_in, n_out))
        is_output = i == len(sizes) - 2
        if not is_output:
            if batch_norm:
                layers.append(nn.BatchNorm1d(n_out))
            layers.append(nn.ReLU())
            if dropout > 0:
                layers.append(nn.Dropout(dropout))
    return nn.Sequential(*layers)


def make_optimizer(model, optimizations):
    if optimizations["optimizer"] == "sgd":
        return optim.SGD(model.parameters(), lr=optimizations["learning_rate"], momentum=0.9,
                         weight_decay=optimizations["weight_decay"])
    return optim.Adam(model.parameters(), lr=optimizations["learning_rate"],
                      weight_decay=optimizations["weight_decay"])


def make_network(hyperparameters):
    """A network is a dict with the model, its hyperparameters and its optimizer."""
    optimizations = hyperparameters["optimizations"]
    model = build_model(hyperparameters["layer_sizes"], optimizations["batch_norm"],
                        optimizations["dropout"]).to(DEVICE)
    return {"model": model, "hyperparameters": hyperparameters,
            "optimizer": make_optimizer(model, optimizations)}

## 4. Entrenar, probar, guardar y cargar

In [ ]:
def train(net, epochs):
    """Trains the network for `epochs` epochs and returns the loss of each one."""
    hyperparameters = net["hyperparameters"]
    labels = get_labels(train_labels, hyperparameters["classification"])
    loader = DataLoader(TensorDataset(train_images, labels), batch_size=BATCH_SIZE, shuffle=True)
    criterion = nn.CrossEntropyLoss()
    losses = []
    net["model"].train()
    for _ in range(epochs):
        total_loss = 0.0
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            net["optimizer"].zero_grad()
            loss = criterion(net["model"](x), y)
            loss.backward()
            net["optimizer"].step()
            total_loss += loss.item() * x.size(0)
        epoch_loss = total_loss / len(loader.dataset)
        losses.append(epoch_loss)
        hyperparameters["epochs"] += 1
        print(f"Epoch {hyperparameters['epochs']}: loss = {epoch_loss:.4f}")
    return losses


def plot_loss(losses, first_epoch):
    epochs = list(range(first_epoch, first_epoch + len(losses)))
    plt.figure(figsize=(7, 4))
    plt.plot(epochs, losses, marker="o")
    plt.xlabel("Época")
    plt.ylabel("Pérdida (loss)")
    plt.title("Pérdida por época")
    plt.grid(True)
    plt.show()


def test(net):
    """Evaluates on the test set: confusion matrix, accuracy, precision and recall (macro)."""
    hyperparameters = net["hyperparameters"]
    class_names = get_class_names(hyperparameters["classification"])
    num_classes = len(class_names)
    labels = get_labels(test_labels, hyperparameters["classification"])
    net["model"].eval()
    with torch.no_grad():
        predictions = net["model"](test_images.to(DEVICE)).argmax(dim=1).cpu()

    confusion = torch.bincount(labels * num_classes + predictions,
                               minlength=num_classes * num_classes).view(num_classes, num_classes)
    true_positives = confusion.diag().double()
    accuracy = true_positives.sum().item() / confusion.sum().item()
    precision = (true_positives / confusion.sum(dim=0).double()).nan_to_num(0.0).mean().item()
    recall = (true_positives / confusion.sum(dim=1).double()).nan_to_num(0.0).mean().item()

    plt.figure(figsize=(7, 6))
    plt.imshow(confusion.numpy(), cmap="Blues")
    plt.colorbar()
    plt.xticks(range(num_classes), class_names, rotation=45, ha="right")
    plt.yticks(range(num_classes), class_names)
    plt.xlabel("Predicción")
    plt.ylabel("Clase real")
    plt.title("Matriz de confusión")
    for i in range(num_classes):
        for j in range(num_classes):
            plt.text(j, i, int(confusion[i, j]), ha="center", va="center", fontsize=8)
    plt.tight_layout()
    plt.show()

    print(f"Accuracy:  {accuracy:.4f}")
    print(f"Precision: {precision:.4f}")
    print(f"Recall:    {recall:.4f}")
    return {"accuracy": accuracy, "precision": precision, "recall": recall}


def save_network(net, path):
    torch.save({"state_dict": net["model"].state_dict(), "hyperparameters": net["hyperparameters"]}, path)
    print(f"Red guardada en {path}")


def load_network(path):
    checkpoint = torch.load(path, map_location=DEVICE)
    state_dict = checkpoint["state_dict"]
    hyperparameters = checkpoint["hyperparameters"]
    # Layer widths come from the shapes of the Linear weights in the state_dict.
    weights = [tensor for tensor in state_dict.values() if tensor.dim() == 2]
    hyperparameters["layer_sizes"] = [weights[0].shape[1]] + [w.shape[0] for w in weights]
    net = make_network(hyperparameters)
    net["model"].load_state_dict(state_dict)
    return net

## 5. Menú

Desde el menú se crea o carga una red, y luego se puede entrenar, probar y guardar. Las
optimizaciones (batch normalization, dropout, optimizador, learning rate y weight decay) se
eligen al crear la red y quedan guardadas en el `.pth`.

In [ ]:
def ask_text(prompt, default):
    value = input(f"{prompt} [{default}]: ").strip()
    return value or str(default)


def ask_option(prompt, options, default):
    """`options` maps each value to its label. Returns the chosen value."""
    values = list(options)
    for number, value in enumerate(values, 1):
        print(f"  {number}) {options[value]}")
    while True:
        choice = ask_text(prompt, values.index(default) + 1)
        if choice.isdigit() and 1 <= int(choice) <= len(values):
            return values[int(choice) - 1]
        print("Opción no válida.")


def ask_number(prompt, default, cast, minimum, maximum=None):
    while True:
        try:
            number = cast(ask_text(prompt, default))
        except ValueError:
            print("Valor no válido.")
            continue
        if number >= minimum and (maximum is None or number <= maximum):
            return number
        print("Valor fuera de rango.")


def create_network():
    print("\n--- Crear nueva red ---")
    network_type = ask_option("Tipo de red", {"rectangular": "Rectangular",
                                              "convolutional": "Convolucional"}, "rectangular")
    classification = ask_option("Clasificación", {"original": "Original (10 clases)",
                                                  "simplified": "Simplificada (4 clases)"}, "original")
    hidden_layers = ask_number("Número de capas ocultas", 2, int, minimum=1)
    neurons_per_layer = None
    if network_type == "rectangular":
        neurons_per_layer = ask_number("Neuronas por capa", 512, int, minimum=1)
    optimizations = {
        "batch_norm": ask_option("¿Usar batch normalization?", {False: "No", True: "Sí"}, False),
        "dropout": ask_number("Probabilidad de dropout (0 = sin dropout)", 0.0, float, 0.0, 0.9),
        "optimizer": ask_option("Optimizador", {"adam": "Adam", "sgd": "SGD"}, "adam"),
        "learning_rate": ask_number("Learning rate", 0.001, float, minimum=1e-6),
        "weight_decay": ask_number("Weight decay", 0.0, float, minimum=0.0),
    }
    num_outputs = NUM_CLASSES_ORIGINAL if classification == "original" else len(CLASS_NAMES_SIMPLIFIED)
    hyperparameters = {
        "network_type": network_type,
        "classification": classification,
        "hidden_layers": hidden_layers,
        "neurons_per_layer": neurons_per_layer,
        "layer_sizes": layer_sizes(network_type, hidden_layers, neurons_per_layer, num_outputs),
        "epochs": 0,
        "optimizations": optimizations,
    }
    print(f"Red creada con capas {hyperparameters['layer_sizes']}")
    return make_network(hyperparameters)


def load_network_from_menu():
    while True:
        path = ask_text("Ruta del archivo .pth", f"pytorch_fashion_{NOTEBOOK_ID}_original_rectangular.pth")
        if os.path.exists(path):
            return load_network(path)
        print(f"No existe el archivo {path}.")


def run_menu():
    net = None
    while True:
        print("\n=== MENÚ ===")
        print("  1) Crear nueva red")
        print("  2) Cargar red desde archivo .pth")
        print("  3) Entrenar la red actual")
        print("  4) Probar la red actual")
        print("  5) Guardar la red actual")
        print("  6) Salir")
        option = input("Seleccione una opción: ").strip()

        if option == "1":
            net = create_network()
        elif option == "2":
            net = load_network_from_menu()
            print(f"Red cargada: {net['hyperparameters']['network_type']}, "
                  f"{net['hyperparameters']['classification']}, "
                  f"épocas acumuladas: {net['hyperparameters']['epochs']}")
        elif option in ("3", "4", "5"):
            if net is None:
                print("Primero cree o cargue una red.")
            elif option == "3":
                epochs = ask_number("Número de épocas", 10, int, minimum=1)
                first_epoch = net["hyperparameters"]["epochs"] + 1
                losses = train(net, epochs)
                plot_loss(losses, first_epoch)
            elif option == "4":
                test(net)
            else:
                hp = net["hyperparameters"]
                default_path = f"pytorch_fashion_{NOTEBOOK_ID}_{hp['classification']}_{hp['network_type']}.pth"
                save_network(net, ask_text("Ruta donde guardar", default_path))
        elif option == "6":
            print("Hasta luego.")
            break
        else:
            print("Opción no válida.")


run_menu()